# M15a · Arch 4A: private angular centers + SB-LN

See `docs4/arch4a_angular_centers.tex`. Private latent = per-generator centers c_g in the angular loss. Deploy discards c_g.

**KEEP** iff NT ≥ 81.21 and DFD ≥ 95.5.


## 1. Setup
### 1.1 Settings


In [1]:
# 1.1 Settings
import json, os
from pathlib import Path

DATA_ROOT = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
FACES_ROOT = DATA_ROOT / "faces"
INDEX_CSV = DATA_ROOT / "index" / "ffpp_c23.csv"
FACES_DFD = DATA_ROOT / "faces_dfd"
DFD_INDEX = DATA_ROOT / "index" / "dfd_c23.csv"
FACES_UADFV = DATA_ROOT / "faces_uadfv"
UADFV_INDEX = DATA_ROOT / "index" / "uadfv.csv"
RUNS_ROOT = DATA_ROOT / "runs"
M15A_DIR = RUNS_ROOT / "m15a"
M11A_DIR = RUNS_ROOT / "m11a"
M11B_DIR = RUNS_ROOT / "m11b"
M4_DIR = RUNS_ROOT / "m4"

ALL_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
TRAIN_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
SHORT = {"original": "real", "Deepfakes": "DF", "Face2Face": "F2F", "FaceSwap": "FS",
         "NeuralTextures": "NT", "FaceShifter": "FSh"}
G = len(TRAIN_GENERATORS)

ARCH, PRETRAINED = "ViT-L-14", "openai"
L2_NORM = True
W_UNIFORM, W_ALIGN = 0.5, 0.1
AUGMENT = True
BATCH_SIZE = 128
EPOCHS = 10
CYCLE_EPOCHS = 10
WARMUP_EPOCHS = 1
LR_MAX, LR_MIN = 3e-4, 1e-5
SEED = 0
RANK = 16
LAMBDA = 0.01
ALPHA_REAL = 0.1
BETA_MARGIN = 1.0
GAMMA_ATTR = 0.5
MARGIN = 0.1
EMA = 0.99
SCALE_INIT = 10.0
PROTO_WARM = 512
RUN_DFD, RUN_UADFV = True, True

A0 = {"seen": 94.54, "DF": 99.49, "F2F": 97.86, "FS": 99.60, "NT": 81.21, "FSh": 92.25, "DFD": 95.15}
REFS = {
    "full LayerNorms (M4 λ=∞)": (M4_DIR / f"gbla_laminf_{ARCH}_seed{SEED}", 96.05, 99.38),
    "GBLA λ=0.01 (M4) / Arch1": (M4_DIR / f"gbla_lam0.01_{ARCH}_seed{SEED}", 96.71, 99.63),
}
VAL_FRAMES_PER_VIDEO = 8
CACHE_IN_RAM = True
EVAL_BATCH = 256
GRAD_CHECKPOINTING = False

COMMON_CONFIG = {
    "architecture": "Arch4A", "notebook": "M15a",
    "arch": ARCH, "rank": RANK, "lambda": LAMBDA,
    "alpha_real": ALPHA_REAL, "beta_margin": BETA_MARGIN, "gamma_attr": GAMMA_ATTR,
    "margin": MARGIN, "ema": EMA, "scale_init": SCALE_INIT,
    "private": "angular centers c_g (train only)",
    "docs": "docs4/arch4a_angular_centers.tex",
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "seed": SEED,
    "lr_max": LR_MAX, "lr_min": LR_MIN, "augment": AUGMENT,
}
assert INDEX_CSV.exists()
assert (M11A_DIR / f"sbln_r{RANK}_{ARCH}_seed{SEED}" / "results.json").exists()
M15A_DIR.mkdir(parents=True, exist_ok=True)
print("M15a ->", M15A_DIR)



M15a -> /nlsasfs/home/isea/isea64/GBLA-Deepfake Detection/runs/m15a


### 1.2 Imports


In [2]:
# 1.2 Imports and environment
import json, math, random, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms import v2 as T
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert torch.cuda.is_available(), "M15a needs the GPU"
DEVICE = "cuda"


def usable_cpus():
    """CPUs this job may use: the affinity mask and the cgroup CPU quota."""
    n = len(os.sched_getaffinity(0))
    for quota_file, period_file in [("/sys/fs/cgroup/cpu.max", None),
                                    ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")]:
        try:
            if period_file is None:
                quota, period = Path(quota_file).read_text().split()
            else:
                quota, period = Path(quota_file).read_text().strip(), Path(period_file).read_text().strip()
            if quota not in ("max", "-1"):
                n = min(n, max(1, int(quota) // int(period)))
        except (OSError, ValueError):
            pass
    return n


def available_memory_gb():
    """Free RAM for this job: the smaller of MemAvailable and the cgroup limit minus non-cache usage."""
    candidates = []
    try:
        meminfo = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
        candidates.append(int(meminfo["MemAvailable"].split()[0]) * 1024)
    except (OSError, KeyError, ValueError):
        pass
    for limit_file, usage_file, stat_file, cache_key in [
            ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.stat", "file"),
            ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes",
             "/sys/fs/cgroup/memory/memory.stat", "cache")]:
        try:
            limit = Path(limit_file).read_text().strip()
            if not limit.isdigit() or int(limit) >= 1 << 60:
                continue
            usage = int(Path(usage_file).read_text().strip())
            stat = dict(line.split() for line in Path(stat_file).read_text().splitlines())
            candidates.append(int(limit) - usage + int(stat.get(cache_key, 0)))
        except (OSError, ValueError):
            pass
    return min(candidates) / 1e9 if candidates else 0.0


CPUS = usable_cpus()
NUM_WORKERS = max(1, min(16, CPUS - 1))     # one CPU stays with the training process
torch.set_num_threads(2)

pd.Series({
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "open_clip": open_clip.__version__,
    "GPU": torch.cuda.get_device_name(0),
    "CPU cores on the node": os.cpu_count(),
    "CPUs usable by this job": CPUS,
    "data loader workers": NUM_WORKERS,
    "free RAM for this job (GB)": round(available_memory_gb(), 1),
}, name="value").to_frame()


/nlsasfs/home/isea/isea64/venv_jupyter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
torch,2.6.0+cu124
torchvision,0.21.0+cu124
open_clip,3.3.0
GPU,NVIDIA A100-SXM4-40GB
CPU cores on the node,256
CPUs usable by this job,1
data loader workers,1
free RAM for this job (GB),973.1


## 2. Data
### 2.1 Index


In [3]:
# 2.1 Index and subsets
index = pd.read_csv(INDEX_CSV, dtype={"video_id": str, "target_id": str, "source_id": str},
                    keep_default_na=False)
CATEGORIES = ["original"] + ALL_GENERATORS
GEN_CODE = {g: k for k, g in enumerate(CATEGORIES)}


def subset(split, generators):
    keep = (index.split == split) & index.generator.isin(["original", *generators])
    return index[keep].reset_index(drop=True)


train_df = subset("train", TRAIN_GENERATORS)
val_all = subset("val", TRAIN_GENERATORS)
frames_per_video = int(val_all.groupby(["generator", "video_id"]).size().max())
keep_every = max(1, frames_per_video // VAL_FRAMES_PER_VIDEO)
frame_rank = val_all.groupby(["generator", "video_id"]).frame_idx.rank(method="first").astype(int) - 1
val_df = val_all[frame_rank % keep_every == 0].reset_index(drop=True)
test_df = subset("test", ALL_GENERATORS)

display(pd.DataFrame({name: df.groupby("generator").size().reindex(CATEGORIES)
                      for name, df in [("train", train_df), ("val (subsampled)", val_df), ("test", test_df)]}
                     ).fillna(0).astype(int))


,train,val (subsampled),test
generator,,,
original,23040,1120,4480
Deepfakes,23040,1120,4480
Face2Face,23040,1120,4480
FaceSwap,23040,1120,4480
NeuralTextures,23040,1120,4480
FaceShifter,0,0,4480


### 2.2 RAM crops


In [4]:
# 2.2 Decoded crops in RAM
def load_images(df, desc):
    """Decodes every crop of an index slice into one uint8 array (N, H, W, 3), using threads."""
    paths = [FACES_ROOT / p for p in df.path]
    first = np.asarray(Image.open(paths[0]).convert("RGB"))
    out = np.empty((len(paths), *first.shape), dtype=np.uint8)

    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))

    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out


images = {}
budget = 0.6 * available_memory_gb()
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    need = len(df) * 224 * 224 * 3 / 1e9
    if CACHE_IN_RAM and need <= budget:
        t = time.time()
        images[name] = load_images(df, f"{name} crops -> RAM")
        budget -= need
        sec = time.time() - t
        print(f"{name}: {len(df):,} crops, {need:.1f} GB in RAM, {sec / 60:.1f} min ({len(df) / sec:.0f} crops/s from disk)")
    else:
        images[name] = None
        print(f"{name}: read from disk during training (needs {need:.1f} GB, RAM budget left {budget:.1f} GB)")


train crops -> RAM: 100%|██████████| 115200/115200 [06:08<00:00, 312.59it/s]


train: 115,200 crops, 17.3 GB in RAM, 6.2 min (309 crops/s from disk)


val crops -> RAM: 100%|██████████| 5600/5600 [00:16<00:00, 341.20it/s]


val: 5,600 crops, 0.8 GB in RAM, 0.3 min (331 crops/s from disk)


test crops -> RAM: 100%|██████████| 26880/26880 [01:15<00:00, 354.48it/s]

test: 26,880 crops, 4.0 GB in RAM, 1.3 min (347 crops/s from disk)


### 2.3 Loader


In [5]:
# 2.3 Round-robin loader
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def make_transform(train):
    steps = []
    if train and AUGMENT:
        steps += [
            T.RandomHorizontalFlip(0.5),
            T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.1),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.JPEG(quality=(40, 100)),
        ]
    steps += [T.ToDtype(torch.float32, scale=True), T.Normalize(CLIP_MEAN, CLIP_STD)]
    return T.Compose(steps)


class FaceDataset(Dataset):
    def __init__(self, df, train, images=None):
        self.paths = [str(FACES_ROOT / p) for p in df.path]
        self.images = images
        self.labels = torch.tensor(df.label.values, dtype=torch.long)
        self.gens = torch.tensor(df.generator.map(GEN_CODE).values, dtype=torch.long)
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.images is not None:
            image = torch.from_numpy(np.array(self.images[i])).permute(2, 0, 1)
        else:
            image = read_image(self.paths[i], mode=ImageReadMode.RGB)
        return self.transform(tv_tensors.Image(image)).as_subclass(torch.Tensor), self.labels[i], self.gens[i]


class RoundRobinSampler(Sampler):
    """One epoch = one block per training generator, in an order fixed by (seed, epoch)."""

    def __init__(self, df, seed):
        self.seed = seed
        self.epoch = 0
        gen = df.generator.values
        real_idx = np.flatnonzero(gen == "original")
        # fixed split: real row r goes to generator r % G
        self.blocks = {}
        for k, g in enumerate(TRAIN_GENERATORS):
            fakes = np.flatnonzero(gen == g)
            reals = real_idx[k::G][: len(fakes) // G]
            self.blocks[g] = (fakes, reals)
        self.n_per = len(self.blocks[TRAIN_GENERATORS[0]][0]) + len(self.blocks[TRAIN_GENERATORS[0]][1])

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        order_rng = np.random.default_rng([self.seed, self.epoch, 1])
        gens = list(TRAIN_GENERATORS)
        order_rng.shuffle(gens)
        out = []
        for g in gens:
            fakes, reals = self.blocks[g]
            rng = np.random.default_rng([self.seed, ALL_GENERATORS.index(g), self.epoch])
            out.extend(rng.permutation(np.concatenate([fakes, reals])).tolist())
        return iter(out)

    def __len__(self):
        return self.n_per * G


train_ds = FaceDataset(train_df, train=True, images=images["train"])
train_sampler = RoundRobinSampler(train_df, SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
owned = {SHORT[g]: (len(f), len(r)) for g, (f, r) in train_sampler.blocks.items()}
print("fakes / reals owned by each generator:", owned)
print(f"steps per epoch: {len(train_loader)}  (M1 was 900);  workers: {NUM_WORKERS}")
assert len(train_loader) == 900, len(train_loader)


fakes / reals owned by each generator: {'DF': (23040, 5760), 'F2F': (23040, 5760), 'FS': (23040, 5760), 'NT': (23040, 5760)}
steps per epoch: 900  (M1 was 900);  workers: 1


### 2.4 Speed


In [6]:
# 2.4 Data-loading speed
def loader_rate(loader, n_batches=30):
    """Steady-state images/s after the pre-fetched batches are used up."""
    it = iter(loader)
    for _ in range(NUM_WORKERS * 4 + 2):
        next(it)
    t = time.time()
    for _ in range(n_batches):
        next(it)
    rate = n_batches * loader.batch_size / (time.time() - t)
    del it
    return rate


t = time.time()
for i in range(200):
    train_ds[i]
ms_per_crop = (time.time() - t) / 200 * 1000
load_speed = loader_rate(train_loader)
GPU_SPEED_M1 = 253
speed = min(load_speed, GPU_SPEED_M1)
pd.Series({
    "CPU time per augmented crop, one process (ms)": round(ms_per_crop, 1),
    "data loading speed (img/s)": round(load_speed),
    "GPU training speed in M1 (img/s)": GPU_SPEED_M1,
    "bottleneck": "GPU" if load_speed >= GPU_SPEED_M1 else "data loading",
    "estimated minutes per epoch (training part)": round(len(train_loader) * BATCH_SIZE / speed / 60, 1),
    f"estimated hours per ablation ({EPOCHS} epochs)": round(EPOCHS * len(train_loader) * BATCH_SIZE / speed / 3600, 1),
}, name="value").to_frame()


,value
"CPU time per augmented crop, one process (ms)",35.2
data loading speed (img/s),155
GPU training speed in M1 (img/s),253
bottleneck,data loading
estimated minutes per epoch (training part),12.4
estimated hours per ablation (10 epochs),2.1


## 3. Model
### 3.1 Detector


In [7]:
# 3.1 Detector (as in M1)
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        encoder = open_clip.create_model(ARCH, pretrained=PRETRAINED, force_quick_gelu=True).visual
        encoder.proj = None                      # keep the 1024-d CLS token
        self.encoder = encoder
        self.head = nn.Linear(encoder.transformer.width, 2)

    def forward(self, x):
        """Returns (logits, L2-normalized features)."""
        feats = self.encoder(x).float()
        f = F.normalize(feats, dim=-1)
        return self.head(f if L2_NORM else feats), f


torch.manual_seed(SEED)
model = Detector().to(DEVICE)
model.requires_grad_(False)
for m in model.encoder.modules():
    if isinstance(m, nn.LayerNorm):
        m.requires_grad_(True)
model.head.requires_grad_(True)
model.encoder.set_grad_checkpointing(GRAD_CHECKPOINTING)

PARAMS = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
PARAM_NAMES = [n for n, _ in PARAMS]
P = sum(p.numel() for _, p in PARAMS)


def get_flat():
    return torch.cat([p.detach().reshape(-1) for _, p in PARAMS]).float()


@torch.no_grad()
def set_flat(theta):
    offset = 0
    for _, p in PARAMS:
        n = p.numel()
        p.copy_(theta[offset:offset + n].view_as(p))
        offset += n


def flat_grad():
    return torch.cat([p.grad.reshape(-1) for _, p in PARAMS]).float()


def flat_from_state(state):
    return torch.cat([state[n].reshape(-1).float() for n in PARAM_NAMES]).to(DEVICE)


def trainable_state():
    return {n: p.detach().cpu().clone() for n, p in PARAMS}


THETA0 = get_flat().clone()
print(f"trainable numbers P = {P:,}")
if ARCH == "ViT-L-14":
    assert P == 104_450


trainable numbers P = 104,450


### 3.2 Arch4A state


In [8]:
# 3.2 Arch4A state: SB-LN + private angular centers
WIDTH = model.encoder.transformer.width
SLICES, offset = {}, 0
for n, p in PARAMS:
    SLICES[n] = slice(offset, offset + p.numel()); offset += p.numel()
LN_MODULES = [n[:-len(".weight")] for n in PARAM_NAMES if n.endswith(".weight") and not n.startswith("head")]
N_LN = len(LN_MODULES)
assert N_LN == 50 and WIDTH == 1024
GAMMA_IDX = torch.stack([torch.arange(SLICES[m + ".weight"].start, SLICES[m + ".weight"].stop) for m in LN_MODULES]).to(DEVICE)
BETA_IDX = torch.stack([torch.arange(SLICES[m + ".bias"].start, SLICES[m + ".bias"].stop) for m in LN_MODULES]).to(DEVICE)
G0, B0 = THETA0[GAMMA_IDX].clone(), THETA0[BETA_IDX].clone()

def sbln_numel(rank=RANK):
    return 2 * WIDTH * rank + 2 * N_LN * rank

def deployed_numel(rank=RANK):
    return sbln_numel(rank) + 1  # + scale s

def train_numel(rank=RANK):
    return deployed_numel(rank) + G * WIDTH  # private centers


class Arch4AState:
    def __init__(self, rank=RANK, seed=SEED):
        self.rank = rank
        gen = torch.Generator().manual_seed(seed)
        scale = math.sqrt(WIDTH / rank)
        def basis():
            q, _ = torch.linalg.qr(torch.randn(WIDTH, rank, generator=gen))
            return (q * scale).to(DEVICE)
        self.p = {"Bg": basis(), "Bb": basis(),
                  "A": torch.zeros(N_LN, rank, device=DEVICE),
                  "C": torch.zeros(N_LN, rank, device=DEVICE),
                  "s": torch.tensor(SCALE_INIT, device=DEVICE)}
        self.centers = {g: F.normalize(torch.randn(WIDTH, generator=gen), dim=0).to(DEVICE)
                        for g in TRAIN_GENERATORS}
        self.proto = {"real": torch.zeros(WIDTH, device=DEVICE), "n_real": 0}

    def theta(self, p=None):
        p = self.p if p is None else p
        t = THETA0.clone()
        t[GAMMA_IDX] = G0 + p["A"] @ p["Bg"].T
        t[BETA_IDX] = B0 + p["C"] @ p["Bb"].T
        return t

    def sb_grads(self, grad):
        Gg, Gb, p = grad[GAMMA_IDX], grad[BETA_IDX], self.p
        return {"A": Gg @ p["Bg"], "Bg": Gg.T @ p["A"], "C": Gb @ p["Bb"], "Bb": Gb.T @ p["C"]}

    def c_real(self):
        assert self.proto["n_real"] > 0
        return F.normalize(self.proto["real"], dim=0)

    def c_priv(self, g):
        return F.normalize(self.centers[g], dim=0)

    @torch.no_grad()
    def update_real(self, z, y):
        real = y == 0
        if real.any():
            new = z[real].mean(0)
            n = self.proto["n_real"]
            self.proto["real"] = new.clone() if n == 0 else EMA * self.proto["real"] + (1 - EMA) * new
            self.proto["n_real"] += 1

    def save(self):
        return {"architecture": "Arch4A", "rank": self.rank,
                "phi": {k: v.detach().cpu() for k, v in self.p.items()},
                "centers": {g: v.detach().cpu() for g, v in self.centers.items()},
                "proto": {"real": self.proto["real"].cpu(), "n_real": self.proto["n_real"]},
                "theta": self.theta().cpu()}

    def load(self, obj):
        for k, v in obj["phi"].items():
            self.p[k].copy_(v.to(DEVICE))
        for g, v in obj["centers"].items():
            self.centers[g].copy_(v.to(DEVICE))
        self.proto["real"].copy_(obj["proto"]["real"].to(DEVICE))
        self.proto["n_real"] = int(obj["proto"]["n_real"])


display(pd.DataFrame([
    {"piece": "SB-LN", "n": sbln_numel(), "dep": True},
    {"piece": "scale s", "n": 1, "dep": True},
    {"piece": "private c_g", "n": G * WIDTH, "dep": False},
    {"piece": "TOTAL deployed", "n": deployed_numel(), "dep": True},
    {"piece": "TOTAL train", "n": train_numel(), "dep": False},
]))
assert deployed_numel() < 104_450
print("deployed", deployed_numel(), "train", train_numel())



,piece,n,dep
0,SB-LN,34368,True
1,scale s,1,True
2,private c_g,4096,False
3,TOTAL deployed,34369,True
4,TOTAL train,38465,False


deployed 34369 train 38465


## 4. Helpers
### 4.1


In [9]:
# 4.1 Helpers
MEAN_GPU = torch.tensor(CLIP_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_GPU = torch.tensor(CLIP_STD, device=DEVICE).view(1, 3, 1, 1)

def encode(x):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        feats = model.encoder(x)
    return F.normalize(feats.float(), dim=-1)

def arch4a_loss(z, y, g, state, s, c_g, margin=MARGIN):
    cr = state.c_real()
    logit = s * (z @ cr)
    logits = torch.stack([torch.zeros_like(logit), logit], dim=1)
    ce_each = F.cross_entropy(logits, y, reduction="none")
    ce = ce_each.mean()
    n = len(y)
    i, j = torch.triu_indices(n, n, offset=1, device=z.device)
    d2 = (2 - 2 * (z @ z.T)[i, j]).clamp(min=0)
    same = y[i] == y[j]
    align = d2[same].mean() if same.any() else d2.new_zeros(())
    uniform = torch.exp(-2 * d2).mean().clamp(min=1e-6).log()
    real = y == 0
    l_real = (1 - (z[real] @ cr)).mean() if real.any() else z.new_zeros(())
    l_margin = F.relu((z[~real] @ cr) - margin).mean() if (~real).any() else z.new_zeros(())
    cg = F.normalize(c_g, dim=0)
    l_attr = (1 - (z[~real] @ cg)).mean() if (~real).any() else z.new_zeros(())
    penalty = LAMBDA * (cg - cr.detach()).pow(2).sum()
    loss = ce + W_ALIGN * align + W_UNIFORM * uniform + ALPHA_REAL * l_real + BETA_MARGIN * l_margin + GAMMA_ATTR * l_attr + penalty
    parts = {"loss": float(loss.detach()), "ce": float(ce.detach()), "attr": float(l_attr.detach()),
             "penalty": float(penalty.detach()), "margin": float(l_margin.detach())}
    return loss, parts, ce_each.detach()

def lr_at(step, steps_per_epoch):
    cycle = CYCLE_EPOCHS * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch
    pos = step % cycle
    if pos < warm:
        return LR_MIN + (LR_MAX - LR_MIN) * pos / warm
    frac = (pos - warm) / max(cycle - warm, 1)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * frac))

def adam_step(param, grad, m, v, t, lr):
    b1, b2, eps = 0.9, 0.999, 1e-8
    t += 1
    m.mul_(b1).add_(grad, alpha=1 - b1)
    v.mul_(b2).addcmul_(grad, grad, value=1 - b2)
    param.addcdiv_(m / (1 - b1 ** t), (v / (1 - b2 ** t)).sqrt().add_(eps), value=-lr)

class AdamSlots:
    @staticmethod
    def tensors(state):
        out = dict(state.p)
        out.update({f"center:{g}": c for g, c in state.centers.items()})
        return out
    def __init__(self, state):
        ts = self.tensors(state)
        self.m = {k: torch.zeros_like(v) for k, v in ts.items()}
        self.v = {k: torch.zeros_like(v) for k, v in ts.items()}
        self.t = {k: torch.zeros((), dtype=torch.long, device=DEVICE) for k in ts}
    def state_dict(self):
        return {"m": {k: v.cpu() for k, v in self.m.items()}, "v": {k: v.cpu() for k, v in self.v.items()},
                "t": {k: int(v) for k, v in self.t.items()}}
    def load_state_dict(self, s):
        for k in self.m:
            self.m[k].copy_(s["m"][k].to(DEVICE)); self.v[k].copy_(s["v"][k].to(DEVICE)); self.t[k].fill_(s["t"][k])

def flat_grad_safe():
    return torch.cat([(p.grad if p.grad is not None else torch.zeros_like(p)).reshape(-1) for _, p in PARAMS]).float()

def train_step(state, opt, x, y, g, lr):
    set_flat(state.theta())
    model.zero_grad(set_to_none=True)
    s = state.p["s"].detach().clone().requires_grad_(True)
    cg = state.centers[g].detach().clone().requires_grad_(True)
    z = encode(x)
    loss, parts, ce_each = arch4a_loss(z, y, g, state, s, cg)
    loss.backward()
    sb = state.sb_grads(flat_grad_safe().detach())
    grads = {**sb, "s": s.grad.detach(), f"center:{g}": cg.grad.detach()}
    state.update_real(z.detach(), y)
    tensors = AdamSlots.tensors(state)
    for k, gk in grads.items():
        adam_step(tensors[k], gk, opt.m[k], opt.v[k], opt.t[k], lr)
    return parts, ce_each

def to_model_input(uint8_batch):
    x = torch.as_tensor(uint8_batch).to(DEVICE, non_blocking=True)
    return (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU

def load_paths(paths, desc):
    out = np.empty((len(paths), 224, 224, 3), dtype=np.uint8)
    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))
    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out

def ram_images(name, df):
    if images.get(name) is None:
        images[name] = load_images(df, f"{name} crops -> RAM")
    return images[name]

@torch.no_grad()
def predict_probs(state, arr, desc="predict"):
    set_flat(state.theta()); model.eval()
    cr, s = state.c_real(), state.p["s"]
    out = []
    for i in tqdm(range(0, len(arr), EVAL_BATCH), desc=desc, leave=False):
        z = encode(to_model_input(arr[i:i + EVAL_BATCH]))
        out.append(torch.sigmoid(s * (z @ cr)).float().cpu())
    model.train()
    return torch.cat(out).numpy()

@torch.no_grad()
def warm_real(state, n_per=PROTO_WARM, seed=SEED):
    arr = ram_images("train", train_df)
    set_flat(state.theta()); model.eval()
    rng = np.random.default_rng([seed, 21])
    rows = np.concatenate([train_sampler.blocks[g][1] for g in TRAIN_GENERATORS])
    if n_per is not None and len(rows) > n_per:
        rows = np.sort(rng.choice(rows, n_per, replace=False))
    total = torch.zeros(WIDTH, device=DEVICE)
    for i in range(0, len(rows), EVAL_BATCH):
        total += encode(to_model_input(arr[rows[i:i + EVAL_BATCH]])).sum(0)
    state.proto["real"] = total / len(rows)
    state.proto["n_real"] = 1
    model.train()

def auc(labels, scores):
    labels = np.asarray(labels)
    ranks = pd.Series(np.asarray(scores)).rank().values
    n_pos, n_neg = labels.sum(), len(labels) - labels.sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def metrics_table(pred):
    groups = {"seen combined (" + "+".join(SHORT[g] for g in TRAIN_GENERATORS) + ")": TRAIN_GENERATORS}
    groups.update({g: [g] for g in ALL_GENERATORS})
    rows = []
    for name, gens in groups.items():
        sub = pred[pred.generator.isin(["original", *gens])]
        if (sub.label == 1).sum() == 0:
            continue
        vid = sub.groupby(["generator", "video_id"], as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
        rows.append({"test set": name, "status": "seen" if set(gens) <= set(TRAIN_GENERATORS) else "UNSEEN",
                     "frame AUC": 100 * auc(sub.label, sub.prob), "video AUC": 100 * auc(vid.label, vid.prob),
                     "video acc @0.5": 100 * float(((vid.prob > 0.5) == vid.label).mean())})
    return pd.DataFrame(rows).round(2)

def video_auc(index_df, probs):
    vid = index_df.assign(prob=probs).groupby("video_id", as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
    return 100 * auc(vid.label, vid.prob)

def frame_pred(df, probs):
    out = df[["split", "label", "generator", "video_id", "frame_idx", "path"]].copy()
    out["prob"] = probs
    return out

def save_atomic(obj, path):
    tmp = path.with_name(path.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)

def load(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def run_dir():
    return M15A_DIR / f"m15a_arch4a_r{RANK}_{ARCH}_seed{SEED}"

def check_config(run_dir, config):
    path = run_dir / "config.json"
    dumped = json.loads(json.dumps(config))
    if path.exists():
        assert json.loads(path.read_text()) == dumped, f"{run_dir} settings differ; delete to restart"
    else:
        path.write_text(json.dumps(dumped, indent=1))



### 4.2 Smoke


In [10]:
# 4.2 Smoke
check_rows = pd.concat([train_df[train_df.generator == "original"].iloc[:4],
                        train_df[train_df.generator == TRAIN_GENERATORS[0]].iloc[:4]])
check_ds = FaceDataset(check_rows.reset_index(drop=True), train=False, images=None)
xs, ys = zip(*[(check_ds[i][0], check_ds[i][1]) for i in range(len(check_ds))])
check_x, check_y = torch.stack(xs).to(DEVICE), torch.stack(ys).to(DEVICE)
g0 = TRAIN_GENERATORS[0]
state = Arch4AState()
warm_real(state, n_per=8)
opt = AdamSlots(state)
before = {k: v.clone() for k, v in AdamSlots.tensors(state).items()}
train_step(state, opt, check_x, check_y, g0, LR_MAX)
after = AdamSlots.tensors(state)
moved = {k: float((before[k] - after[k]).abs().max()) > 0 for k in before}
assert moved["A"] and moved["C"] and moved["s"] and moved[f"center:{g0}"]
assert not moved["Bg"] and not moved["Bb"]
print("M15a SMOKE OK", {k: moved[k] for k in ("A", "C", "s", f"center:{g0}")})
del state, opt; torch.cuda.empty_cache()



M15a SMOKE OK {'A': True, 'C': True, 's': True, 'center:Deepfakes': True}


## 5. Train


In [11]:
# 5.1 Train
def train_arch4a():
    rd = run_dir(); rd.mkdir(parents=True, exist_ok=True)
    check_config(rd, {**COMMON_CONFIG, "deployed_trainable": deployed_numel(), "train_trainable": train_numel()})
    state, opt, history, start_epoch = Arch4AState(), None, [], 0
    if (rd / "last.pt").exists():
        ckpt = load(rd / "last.pt"); state.load(ckpt["state"])
        opt = AdamSlots(state); opt.load_state_dict(ckpt["adam"])
        history, start_epoch = ckpt["history"], ckpt["epoch"]
        print("resumed", start_epoch)
    else:
        warm_real(state); opt = AdamSlots(state)
    print(f"===== Arch4A r={RANK} deployed={deployed_numel():,} {start_epoch}/{EPOCHS} =====")
    steps_per_epoch = len(train_loader)
    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch); model.train()
        sums, n_steps, t0 = defaultdict(float), 0, time.time()
        ce_sum = torch.zeros(len(CATEGORIES), device=DEVICE)
        ce_cnt = torch.zeros(len(CATEGORIES), device=DEVICE)
        for i, (x, y, gen) in enumerate(tqdm(train_loader, desc=f"4A ep {epoch+1}/{EPOCHS}", leave=False)):
            x, y, gen = (t.to(DEVICE, non_blocking=True) for t in (x, y, gen))
            fake_codes = gen[gen != GEN_CODE["original"]]
            assert len(fake_codes) > 0 and torch.all(fake_codes == fake_codes[0])
            g = CATEGORIES[int(fake_codes[0])]
            lr = lr_at(epoch * steps_per_epoch + i, steps_per_epoch)
            parts, ce_each = train_step(state, opt, x, y, g, lr)
            for k, v in parts.items(): sums[k] += v
            ce_sum += torch.bincount(gen, weights=ce_each, minlength=len(CATEGORIES))
            ce_cnt += torch.bincount(gen, minlength=len(CATEGORIES)); n_steps += 1
        val_table = metrics_table(frame_pred(val_df, predict_probs(state, ram_images("val", val_df), "val")))
        row = {"epoch": epoch + 1, "lr (end)": float(lr)}
        row.update({k: v / n_steps for k, v in sums.items()})
        row.update({f"CE {SHORT[c]}": float(ce_sum[k] / ce_cnt[k]) for k, c in enumerate(CATEGORIES) if ce_cnt[k] > 0})
        row.update({f"|c| {SHORT[g]}": float(state.centers[g].norm()) for g in TRAIN_GENERATORS})
        row.update({"s": float(state.p["s"]), "val video AUC": float(val_table["video AUC"].iloc[0]),
                    "minutes": (time.time() - t0) / 60})
        history.append(row)
        save_atomic(state.save(), rd / f"epoch_{epoch+1:02d}.pt")
        save_atomic({"epoch": epoch + 1, "state": state.save(), "adam": opt.state_dict(), "history": history}, rd / "last.pt")
        print(" | ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in row.items()))
    if not (rd / "results.json").exists():
        # refresh real centroid
        warm_real(state, n_per=None)
        table = metrics_table(frame_pred(test_df, predict_probs(state, ram_images("test", test_df), "test")))
        display(table)
        res = {"architecture": "Arch4A", "notebook": "M15a", "deployed_trainable": deployed_numel(),
               "train_trainable": train_numel(), "test": table.to_dict(orient="records"), "history": history,
               "center_norms": {SHORT[g]: float(state.centers[g].norm()) for g in TRAIN_GENERATORS},
               "s": float(state.p["s"])}
        for name, flag, index_path, root in [("dfd", RUN_DFD, DFD_INDEX, FACES_DFD), ("uadfv", RUN_UADFV, UADFV_INDEX, FACES_UADFV)]:
            if not flag: continue
            idx = pd.read_csv(index_path, dtype={"video_id": str})
            arr = load_paths([root / p for p in idx.path], f"{name} crops")
            res[f"{name}_video_auc"] = video_auc(idx, predict_probs(state, arr, name)); del arr
            print(name.upper(), res[f"{name}_video_auc"])
        (rd / "results.json").write_text(json.dumps(res, indent=1))
    del state, opt; torch.cuda.empty_cache()

train_arch4a(); print("M15a finished")



===== Arch4A r=16 deployed=34,369 0/10 =====


epoch 1 | lr (end) 0.0002997 | loss 0.5007 | ce 0.9486 | attr 0.878 | penalty 0.01458 | margin 0.35 | CE real 4.553 | CE DF 0.1581 | CE F2F 0.004608 | CE FS 0.02559 | CE NT 0.001947 | |c| DF 0.9683 | |c| F2F 0.9587 | |c| FS 0.9804 | |c| NT 0.9812 | s 9.928 | val video AUC 36.38 | minutes 17.07


epoch 2 | lr (end) 0.0002913 | loss -0.665 | ce 0.5277 | attr 0.8921 | penalty 0.007004 | margin 0.02014 | CE real 1.466 | CE DF 0.2831 | CE F2F 0.2974 | CE FS 0.2973 | CE NT 0.2949 | |c| DF 0.9261 | |c| F2F 1.028 | |c| FS 0.9994 | |c| NT 1.025 | s 9.939 | val video AUC 43.98 | minutes 16.45


epoch 3 | lr (end) 0.0002661 | loss -0.6941 | ce 0.5159 | attr 0.8781 | penalty 0.003031 | margin 0.01858 | CE real 1.412 | CE DF 0.2947 | CE F2F 0.2911 | CE FS 0.2895 | CE NT 0.2917 | |c| DF 0.942 | |c| F2F 1.044 | |c| FS 1.018 | |c| NT 1.045 | s 9.967 | val video AUC 50.6 | minutes 16.61


epoch 4 | lr (end) 0.0002275 | loss -0.7053 | ce 0.5089 | attr 0.8737 | penalty 0.00209 | margin 0.01829 | CE real 1.39 | CE DF 0.2901 | CE F2F 0.2872 | CE FS 0.2862 | CE NT 0.2915 | |c| DF 0.9514 | |c| F2F 1.07 | |c| FS 1.021 | |c| NT 1.043 | s 10.01 | val video AUC 54.71 | minutes 15.03


epoch 5 | lr (end) 0.0001802 | loss -0.7136 | ce 0.5017 | attr 0.8709 | penalty 0.001849 | margin 0.01862 | CE real 1.366 | CE DF 0.2821 | CE F2F 0.2887 | CE FS 0.2836 | CE NT 0.2883 | |c| DF 0.9588 | |c| F2F 1.08 | |c| FS 1.022 | |c| NT 1.048 | s 10.08 | val video AUC 59.18 | minutes 12.78


... (6 outputs left out)

dfd crops: 100%|██████████| 109454/109454 [06:45<00:00, 270.11it/s]
                                                      

DFD 45.46873260278499


uadfv crops: 100%|██████████| 3136/3136 [00:08<00:00, 352.86it/s]
                                                      

UADFV 40.9829237817576
M15a finished


## 6. Decision


In [12]:
# 6.1 Decision
def video_auc_from_records(records):
    t = pd.DataFrame(records).set_index("test set")
    row = {"seen": float(t.iloc[0]["video AUC"])}
    row.update({SHORT[g]: float(t.loc[g, "video AUC"]) for g in ALL_GENERATORS})
    return row

rows = [{"model": "A0 SB-LN+linear", "trainable": 36418, **A0, "UADFV": float("nan")}]
for name, (folder, dfd, uadfv) in REFS.items():
    if not (folder / "results.json").exists():
        print("missing", name); continue
    res = json.loads((folder / "results.json").read_text())
    rows.append({"model": name, "trainable": res.get("deployed_trainable", P),
                 **video_auc_from_records(res["test"]),
                 "DFD": res.get("dfd_video_auc", dfd), "UADFV": res.get("uadfv_video_auc", uadfv)})
res = json.loads((run_dir() / "results.json").read_text())
arch = {"model": "Arch4A (M15a)", "trainable": res["deployed_trainable"],
        **video_auc_from_records(res["test"]),
        "DFD": res.get("dfd_video_auc", float("nan")), "UADFV": res.get("uadfv_video_auc", float("nan"))}
rows.append(arch)
main_table = pd.DataFrame(rows).set_index("model"); display(main_table.round(2))
main_table.to_csv(M15A_DIR / "main_table.csv")
keep = float(arch["NT"]) >= A0["NT"] and float(arch["DFD"]) >= 95.5
print(f"NT={arch['NT']:.2f} DFD={arch['DFD']:.2f}")
print("DECISION:", "KEEP Arch4A" if keep else "REJECT Arch4A")
(M15A_DIR / "decision.json").write_text(json.dumps({
    "architecture": "Arch4A", "keep": keep,
    "arch": {k: arch[k] for k in ("seen", "NT", "FSh", "DFD", "trainable")},
    "delta_vs_A0": {k: round(float(arch[k]) - A0[k], 2) for k in ("seen", "NT", "FSh", "DFD")},
}, indent=2))



,trainable,seen,DF,F2F,FS,NT,FSh,DFD,UADFV
model,,,,,,,,,
A0 SB-LN+linear,36418,94.54,99.49,97.86,99.60,81.21,92.25,95.15,NaN
full LayerNorms (M4 λ=∞),104450,96.41,99.72,98.66,99.34,87.90,90.53,96.05,99.38
GBLA λ=0.01 (M4) / Arch1,104450,95.55,99.43,98.55,99.17,85.06,92.91,96.71,99.63
Arch4A (M15a),34369,43.23,37.15,48.03,41.48,46.28,48.37,45.47,40.98


NT=46.28 DFD=45.47
DECISION: REJECT Arch4A


270